In [4]:
import json
import pandas as pd
import glob

In [ ]:
# global topic number: g<grade>t<topic_num>

def assign_topic_number(uid, topic_num):
    if int(uid) <=  8 or int(uid) >= 17:
        grade = "3"
    return "g"+grade+"t"+str(topic_num)

In [5]:
def get_query_actions(log_entries):
    query_actions = []
    for entry in log_entries:
        if entry["type"] == "querySubmitted":
            query_actions.append([entry["uid"], entry["actual_topic_number"], entry["rawQuery"], "UserFormulated", entry["timestamp"]])
        elif entry["type"] == "choseAutoCompleteSuggestion":
            query_actions.append([entry["uid"], entry["actual_topic_number"], entry["selectedSuggestion"], "AutoCompleteSuggestion", entry["timestamp"]])
        elif entry["type"] == "clickedDidYouMeanSuggestion":
            query_actions.append([int(entry["uid"]), entry["actual_topic_number"], entry["suggested query"], "QueryCorrection", entry["timestamp"]])
        elif entry["type"] == "taskQuestionHelpClicked":
            query_actions.append([int(entry["uid"]), entry["actual_topic_number"], None, "TaskQuestionHelp", entry["timestamp"]])


    query_actions_df = pd.DataFrame(query_actions, columns = ["uid", "topic_number", "query", "query_source", "timestamp"])
    return query_actions_df

In [ ]:
def get_generated_serps(log_entries):
    generated_serps = []
    for entry in log_entries:
        if entry["type"] == "searchResultGenerated":
            generated_serps.append([int(entry["uid"]), entry["actual_topic_number"], entry["query"], entry["url"], entry["title"], entry["snippet"], entry["rank"], entry["page"], entry["timestamp"]])
        elif entry["type"] == "searchNoResults":
            generated_serps.append([int(entry["uid"]), entry["actual_topic_number"], entry["query"], None, None, None, None, None, entry["timestamp"]])

    generated_serp_df = pd.DataFrame(generated_serps, columns = ["uid", "topic_number", "query", "url", "title", "snippet", "page_rank", "page_number", "timestamp"])
    return generated_serp_df

In [7]:
def get_clicked_results(log_entries):
    clicked_results = []
    for entry in log_entries:
        if entry["type"] == "clickedResult":
            clicked_results.append([int(entry["uid"]), entry["actual_topic_number"], entry["query"], entry["url"], entry["rank"], entry["page"], entry["timestamp"]])

    clicked_results_df = pd.DataFrame(clicked_results, columns = ["uid", "topic_number", "query", "url", "page_rank", "page_number", "timestamp"])
    return clicked_results_df

In [8]:
query_actions_df = pd.DataFrame() 
generated_serp_df = pd.DataFrame()
clicked_results_df = pd.DataFrame()

for file_name in glob.glob("Data/trial/*.log"):
    # print(file_name)
    with open(file_name, 'r') as f:
        result = [json.loads(x) for x in f.readlines()]
    
    # query actions
    query_temp_df = get_query_actions(result)
    query_actions_df = pd.concat([query_actions_df, query_temp_df], ignore_index=True)

    # SERPs generated
    serp_temp_df = get_generated_serps(result)
    generated_serp_df = pd.concat([generated_serp_df, serp_temp_df], ignore_index=True)

    # Clicked results
    clicked_temp_df = get_clicked_results(result)
    clicked_results_df = pd.concat([clicked_results_df, clicked_temp_df], ignore_index=True)

In [9]:
generated_serp_df.head()

,uid,topic_number,query,url,title,snippet,page_rank,page_number,timestamp
0,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.focusjunior.it/scienza/spazio/stel...,10 cose (+3) che forse non sai sulle stelle - ...,"\n Informazioni scientifiche, miti e ...",1,1,2026-05-19T10:44:36.488
1,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://123stella.astronomiamo.it/Fase.aspx?Ar...,Cosa sono le galassie - 123Stella,"\n Oct 19, 2015 ... IL NOME DELLA VIA...",2,1,2026-05-19T10:44:36.489
2,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.castfvg.it/galassie/gruppo_locale/...,Il Gruppo Locale,\n Le altre galassie del gruppo di M ...,3,1,2026-05-19T10:44:36.491
3,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.focusjunior.it/content/uploads/sit...,"SP ""Aldo Moro"" - Focus Junior","\n May 30, 2016 ... Le stelle sono og...",4,1,2026-05-19T10:44:36.492
4,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.focusjunior.it/scienza/spazio/mari...,Mari spaziali: come sono gli oceani sugli altr...,"\n Giusta distanza dal Sole, gravità,...",5,1,2026-05-19T10:44:36.493


### Exploring data

In [7]:
query_actions_df

,uid,topic_number,query,query_source,timestamp
0,10,3,Quali sono i nomi di altre 3 galassie oltre la...,UserFormulated,2026-05-19T10:44:35.207
1,10,2,qual è,AutoCompleteSuggestion,2026-05-19T10:53:50.940
2,10,2,qual'è stato il primo animale nello spazio,AutoCompleteSuggestion,2026-05-19T10:55:02.986
3,10,2,qual'è stato il primo animale nello spazio e c...,UserFormulated,2026-05-19T10:55:57.938
4,10,1,Come si chiamano i satelliti maggiori di Giove,UserFormulated,2026-05-19T11:02:08.749
5,10,1,Come si chiamano i satelliti maggiori di Giove,UserFormulated,2026-05-19T11:04:29.983
6,2,3,Quali sono i animali dall'India,UserFormulated,2026-05-19T14:50:22.081
7,2,3,Quali sono i 5 animali tipici dall'India,UserFormulated,2026-05-19T14:51:20.189
8,2,2,Qual è il nome del simbolo al centro della ban...,UserFormulated,2026-05-19T15:05:41.521
9,2,2,Qual è il nome del simbolo al centro della b...,UserFormulated,2026-05-19T15:09:06.317


In [8]:
list(query_actions_df["uid"].unique())

['10', '2', '5', '9']

In [9]:
user_topics = pd.read_csv("Data/user_topics.csv")
user_topics = user_topics[["uid", "topic1_question", "topic2_question", "topic3_question"]]
user_topics = user_topics.rename(columns={"topic1_question": "topic1", "topic2_question": "topic2", "topic3_question": "topic3"})
user_topics = user_topics.astype({"uid":"str"})
user_topics.dtypes

uid       object
topic1    object
topic2    object
topic3    object
dtype: object

In [10]:
selected_user_topics = user_topics.loc[user_topics["uid"].isin(list(query_actions_df["uid"].unique()))]

In [11]:
for uid, topic_num in zip(query_actions_df["uid"], query_actions_df["topic_number"]):
    prompt = user_topics.loc[user_topics["uid"]==uid]["topic"+str(topic_num)]
    print(prompt.values[0])

Quali sono i nomi di altre 3 galassie oltre la nostra?
Qual è stato il primo animale nello spazio e come si chiamava?
Qual è stato il primo animale nello spazio e come si chiamava?
Qual è stato il primo animale nello spazio e come si chiamava?
Come si chiamano i satelliti maggiori di Giove?
Come si chiamano i satelliti maggiori di Giove?
Quali sono 5 animali tipici dell'India?
Quali sono 5 animali tipici dell'India?
Qual è il nome del simbolo al centro della bandiera dell'India, e che cosa rappresenta?
Qual è il nome del simbolo al centro della bandiera dell'India, e che cosa rappresenta?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Come si chiama il fiume sacro agli indiani?
Quali sono 5 animali tipici dell'India?
Quali sono 5

In [12]:
query_actions_df["prompt"] = [user_topics.loc[user_topics["uid"]==uid]["topic"+str(topic_num)].values[0] for uid, topic_num in zip(query_actions_df["uid"], query_actions_df["topic_number"])]
query_actions_df

,uid,topic_number,query,query_source,timestamp,prompt
0,10,3,Quali sono i nomi di altre 3 galassie oltre la...,UserFormulated,2026-05-19T10:44:35.207,Quali sono i nomi di altre 3 galassie oltre la...
1,10,2,qual è,AutoCompleteSuggestion,2026-05-19T10:53:50.940,Qual è stato il primo animale nello spazio e c...
2,10,2,qual'è stato il primo animale nello spazio,AutoCompleteSuggestion,2026-05-19T10:55:02.986,Qual è stato il primo animale nello spazio e c...
3,10,2,qual'è stato il primo animale nello spazio e c...,UserFormulated,2026-05-19T10:55:57.938,Qual è stato il primo animale nello spazio e c...
4,10,1,Come si chiamano i satelliti maggiori di Giove,UserFormulated,2026-05-19T11:02:08.749,Come si chiamano i satelliti maggiori di Giove?
5,10,1,Come si chiamano i satelliti maggiori di Giove,UserFormulated,2026-05-19T11:04:29.983,Come si chiamano i satelliti maggiori di Giove?
6,2,3,Quali sono i animali dall'India,UserFormulated,2026-05-19T14:50:22.081,Quali sono 5 animali tipici dell'India?
7,2,3,Quali sono i 5 animali tipici dall'India,UserFormulated,2026-05-19T14:51:20.189,Quali sono 5 animali tipici dell'India?
8,2,2,Qual è il nome del simbolo al centro della ban...,UserFormulated,2026-05-19T15:05:41.521,Qual è il nome del simbolo al centro della ban...
9,2,2,Qual è il nome del simbolo al centro della b...,UserFormulated,2026-05-19T15:09:06.317,Qual è il nome del simbolo al centro della ban...


In [13]:
query_actions_df["query_source"].value_counts()

query_source
UserFormulated            32
AutoCompleteSuggestion     4
Name: count, dtype: int64

In [14]:
clicked_results_df

,uid,topic_number,query,url,page_rank,page_number,timestamp
0,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.focusjunior.it/news/i-musei-scient...,10,1,2026-05-19T10:45:07.398
1,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.focusjunior.it/scienza/spazio/stel...,1,1,2026-05-19T10:46:34.863
2,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://123stella.astronomiamo.it/Fase.aspx?Ar...,2,1,2026-05-19T10:47:32.166
3,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.castfvg.it/galassie/gruppo_locale/...,3,1,2026-05-19T10:48:27.880
4,10,2,qual'è stato il primo animale nello spazio e c...,https://www.storicang.it/foto-del-giorno/laika...,1,1,2026-05-19T10:56:19.184
5,10,2,qual'è stato il primo animale nello spazio e c...,https://www.storicang.it/foto-del-giorno/laika...,1,1,2026-05-19T10:57:54.624
6,2,3,Quali sono i 5 animali tipici dall'India,https://wwftravel.it/blog/viaggi/india-natura-...,1,1,2026-05-19T14:52:53.759
7,2,2,Qual è il nome del simbolo al centro della b...,https://sullestradedelmondo.it/bandiera-indiana/,3,1,2026-05-19T15:09:25.575
8,2,1,Come si chiama il fiume sacro agli indiani,https://www.consciousjourneys.com/it/approfond...,1,1,2026-05-19T15:14:18.095
9,5,3,Quali sono 5 animali tipici dell india,https://www.terreincognitemagazine.it/natura-s...,2,1,2026-05-19T15:08:18.588


In [15]:
clicked_results_df["page_number"].value_counts()

page_number
1    44
Name: count, dtype: int64

In [16]:
print(generated_serp_df.loc[generated_serp_df["uid"]==5])

     uid topic_number                    query  \
71     5            1  Come si chiama il fiume   
72     5            1  Come si chiama il fiume   
73     5            1  Come si chiama il fiume   
74     5            1  Come si chiama il fiume   
75     5            1  Come si chiama il fiume   
..   ...          ...                      ...   
147    5            2                 Bandiera   
148    5            2                 Bandiera   
149    5            2                 Bandiera   
150    5            2                 Bandiera   
151    5            2                 Bandiera   

                                                   url  \
71   https://www.goasia.it/la-religione-gange-fiume...   
72   https://www.focusjunior.it/scuola/storia/dove-...   
73   https://www.beyondthetrip.net/2019/04/03/fiume...   
74   https://www.consciousjourneys.com/it/approfond...   
75   https://isha.sadhguru.org/mahashivratri/it/shi...   
..                                                 

In [17]:
generated_serp_df.loc[generated_serp_df["uid"]==5].to_csv("Data/u5_SERPS.csv", index=False)

In [25]:
generated_serp_df.loc[generated_serp_df["page_rank"].isin(["1", None])]

,uid,topic_number,query,url,title,snippet,page_rank,page_number,timestamp
0,10,3,Quali sono i nomi di altre 3 galassie oltre la...,https://www.focusjunior.it/scienza/spazio/stel...,10 cose (+3) che forse non sai sulle stelle - ...,"\n Informazioni scientifiche, miti e ...",1,1,2026-05-19T10:44:36.488
10,10,2,qual'è stato il primo animale nello spazio e c...,https://www.storicang.it/foto-del-giorno/laika...,"Laika, il primo animale nello spazio - Storica...","\n Nov 3, 2022 ... È probabilmente la...",1,1,2026-05-19T10:55:58.959
20,10,1,Come si chiamano i satelliti maggiori di Giove,https://www.astronomitaly.com/blog/astronomy-p...,"Io, Europa, Ganimede e Callisto: guida ai 4 sa...","\n Mar 19, 2024 ... Io, Europa, Ganim...",1,1,2026-05-19T11:02:09.754
26,10,1,Come si chiamano i satelliti maggiori di Giove,https://www.astronomitaly.com/blog/astronomy-p...,"Io, Europa, Ganimede e Callisto: guida ai 4 sa...","\n Mar 19, 2024 ... Io, Europa, Ganim...",1,1,2026-05-19T11:04:31.470
32,2,3,Quali sono i animali dall'India,https://www.focusjunior.it/scuola/storia/il-pi...,Chi ha inventato il pigiama? - FocusJunior.it,\n Ma quali sono le sue origini? DAL ...,1,1,2026-05-19T14:50:23.230
42,2,3,Quali sono i 5 animali tipici dall'India,https://wwftravel.it/blog/viaggi/india-natura-...,"India naturalistica, 5 animali da non perdere ...","\n Oct 31, 2023 ... La fauna selvatic...",1,1,2026-05-19T14:51:21.413
52,2,2,Qual è il nome del simbolo al centro della ban...,https://mastroviaggiatore.it/saree/blog/lindia...,"L'INDIA A COLORI, I COLORI DELL'INDIA - MASTRO...","\n Feb 14, 2025 ... ... e le interazi...",1,1,2026-05-19T15:05:43.123
58,2,2,Qual è il nome del simbolo al centro della b...,https://mastroviaggiatore.it/saree/blog/lindia...,"L'INDIA A COLORI, I COLORI DELL'INDIA - MASTRO...","\n Feb 14, 2025 ... L'elemento più im...",1,1,2026-05-19T15:09:07.969
64,2,1,Come si chiama il fiume sacro agli indiani,https://www.consciousjourneys.com/it/approfond...,"Gange, il fiume sacro agli indù e alla popolaz...","\n È altresì importante per la pesca,...",1,1,2026-05-19T15:14:11.974
71,5,1,Come si chiama il fiume,https://www.goasia.it/la-religione-gange-fiume...,"La religione e il Gange, il fiume sacro dell'I...",\n ... si dice possa curare anche i m...,1,1,2026-05-19T14:50:32.173
